# Pareto Incentive Group Experiment

This notebook optimizes federal incentive strategies where ZIP codes are partitioned into `n` groups,
and each group receives one incentive amount.

It reuses cached ZIP payloads from `Examples/model_cache/federal` to avoid recomputing agent models.

In [ ]:
from pathlib import Path
import json
import pickle
import random
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

try:
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False
    print("sklearn unavailable; install scikit-learn for clustering")

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "Models").exists():
    repo_root = repo_root.parent

if not (repo_root / "Models").exists():
    raise RuntimeError("Could not locate repository root containing 'Models' directory.")

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Data.data_load_util import make_dataset
from Simulation.projections_util import create_paper_objectives

plt.style.use("seaborn-v0_8")

In [ ]:
# --- Experiment configuration ---
N_VALUES = [1, 2, 3, 5, 8]
INCENTIVE_GRID = list(range(-2000, 8001, 500))
TARGET_ADOPTION_MODE = "additional_only"  # or "absolute_total"
YEARS_TO_SIMULATE = 5
REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION = 0

# Weighted-sweep stage
WEIGHT_SAMPLES = 120
LOCAL_SEARCH_STEPS = 80
LOCAL_SEARCH_NEIGHBORS = 6

# Pareto refinement stage
REFINE_POP_SIZE = 120
REFINE_GENERATIONS = 35
REFINE_MUTATION_RATE = 0.25

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

FEDERAL_PAYLOAD_CACHE_DIR = repo_root / "Examples" / "model_cache" / "federal"
EXPERIMENT_CACHE_DIR = repo_root / "Examples" / "model_cache" / "pareto_experiments"
EXPERIMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = repo_root / "Examples" / "pareto_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not FEDERAL_PAYLOAD_CACHE_DIR.exists():
    raise RuntimeError("Missing federal payload cache directory. Run federal incentive simulation cache build first.")

print(f"Using federal cache: {FEDERAL_PAYLOAD_CACHE_DIR}")
print(f"Writing experiment artifacts to: {EXPERIMENT_CACHE_DIR}")

In [ ]:
state_behavior_df = pd.read_csv(repo_root / "Models" / "Incentives" / "state_behavior.csv")
zips_df, _, _ = make_dataset(granularity="both", remove_outliers=False, load_dir_prefix=str(repo_root / "Data") + "/")
zip_lookup = zips_df.set_index("region_name")
objectives = create_paper_objectives()

def build_target_adoption_curve(base_adoption, annual_increment, years_total, mode):
    targets = []
    for year in range(1, years_total + 1):
        if mode == "absolute_total":
            target = base_adoption + year * annual_increment
        elif mode == "additional_only":
            target = year * annual_increment
        else:
            raise ValueError(f"Unknown TARGET_ADOPTION_MODE: {mode}")
        targets.append(float(np.clip(target, 0.0, 1.0)))
    return targets

def required_incentives_from_payload(payload, cutoff):
    return payload["installation_cost"] - payload["yearly_savings_coeff"] * cutoff

def evaluate_cutoff_adoption_payloads(payloads, cutoff, incentive_threshold):
    zip_adoptions = []
    for payload in payloads:
        required = required_incentives_from_payload(payload, cutoff)
        zip_adoptions.append(float(np.mean(required <= incentive_threshold)))
    return float(np.mean(zip_adoptions)) if zip_adoptions else 0.0

def calibrate_data_based_cutoffs(payloads, yearly_targets, incentive_threshold, lower=0.25, upper=30.0, steps=30):
    calibrated_cutoffs = []
    achieved_adoptions = []
    prev_cutoff = lower

    for target in yearly_targets:
        lo = prev_cutoff
        hi = upper

        upper_adopt = evaluate_cutoff_adoption_payloads(payloads, hi, incentive_threshold)
        if upper_adopt < target:
            calibrated_cutoffs.append(hi)
            achieved_adoptions.append(upper_adopt)
            prev_cutoff = hi
            continue

        lower_adopt = evaluate_cutoff_adoption_payloads(payloads, lo, incentive_threshold)
        if lower_adopt >= target:
            calibrated_cutoffs.append(lo)
            achieved_adoptions.append(lower_adopt)
            prev_cutoff = lo
            continue

        for _ in range(steps):
            mid = 0.5 * (lo + hi)
            mid_adopt = evaluate_cutoff_adoption_payloads(payloads, mid, incentive_threshold)
            if mid_adopt < target:
                lo = mid
            else:
                hi = mid

        final_cutoff = hi
        final_adopt = evaluate_cutoff_adoption_payloads(payloads, final_cutoff, incentive_threshold)
        calibrated_cutoffs.append(final_cutoff)
        achieved_adoptions.append(final_adopt)
        prev_cutoff = final_cutoff

    return calibrated_cutoffs, achieved_adoptions

In [ ]:
def load_state_payloads_from_cache(cache_dir):
    state_payloads = {}
    for p in sorted(cache_dir.glob("zip_payloads_*.pkl")):
        parts = p.stem.split("_")
        if len(parts) < 3:
            continue
        state_code = parts[2]
        with open(p, "rb") as f:
            cached = pickle.load(f)
        payloads = cached.get("payloads", [])
        if payloads:
            # Keep largest cache variant for the state.
            if state_code not in state_payloads or len(payloads) > len(state_payloads[state_code]):
                state_payloads[state_code] = payloads
    return state_payloads

state_payloads = load_state_payloads_from_cache(FEDERAL_PAYLOAD_CACHE_DIR)
print(f"Loaded payloads for {len(state_payloads)} states")

state_cutoffs = {}
state_cutoff_fit_rows = []

for state_code in tqdm(sorted(state_payloads.keys()), desc="Calibrating state cutoffs", unit="state"):
    row = state_behavior_df[state_behavior_df["State code"] == state_code]
    if row.empty:
        continue
    row = row.iloc[0]

    target_curve = build_target_adoption_curve(
        float(row["prop_adopted_status_quo"]),
        float(row["prop_adopted_per_year_average"]),
        YEARS_TO_SIMULATE,
        TARGET_ADOPTION_MODE,
    )

    ref_threshold = -REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION
    cutoffs, achieved = calibrate_data_based_cutoffs(state_payloads[state_code], target_curve, ref_threshold)
    state_cutoffs[state_code] = cutoffs

    for year in range(1, YEARS_TO_SIMULATE + 1):
        state_cutoff_fit_rows.append(
            {
                "state_code": state_code,
                "year": year,
                "target_adoption": target_curve[year - 1],
                "achieved_adoption": achieved[year - 1],
                "cutoff": cutoffs[year - 1],
            }
        )

state_cutoff_fit_df = pd.DataFrame(state_cutoff_fit_rows)
display(state_cutoff_fit_df.head())

In [ ]:
zip_rows = []
for state_code, payloads in state_payloads.items():
    if state_code not in state_cutoffs:
        continue
    cutoff_final = state_cutoffs[state_code][-1]
    for payload in payloads:
        zip_code = int(payload["zip"])
        if zip_code not in zip_lookup.index:
            continue
        z = zip_lookup.loc[zip_code]
        coeff = payload["yearly_savings_coeff"]
        zip_rows.append(
            {
                "zip": zip_code,
                "state_code": state_code,
                "num_agents": int(payload["num_agents"]),
                "installation_cost": float(payload["installation_cost"]),
                "count_qualified": float(z["count_qualified"]),
                "sunlight": float(z["yearly_sunlight_kwh_kw_threshold_avg"]),
                "carbon_per_panel": float(z["carbon_offset_metric_tons_per_panel"]),
                "median_income": float(z["Median_income"]),
                "black_prop": float(z["black_prop"]),
                "panel_utilization": float(z["panel_utilization"]),
                "coeff_mean": float(np.mean(coeff)),
                "coeff_std": float(np.std(coeff)),
                "coeff_p50": float(np.percentile(coeff, 50)),
                "coeff_p90": float(np.percentile(coeff, 90)),
                "state_cutoff_final": float(cutoff_final),
                "payload": payload,
            }
        )

zip_df = pd.DataFrame(zip_rows)
print(f"Working ZIPs: {len(zip_df)}")
display(zip_df.head())

In [ ]:
def cluster_zip_codes_for_n(zip_feature_df, n_groups, cache_dir, random_state=42):
    if not SKLEARN_OK:
        raise RuntimeError("scikit-learn is required for clustering.")

    cache_path = cache_dir / f"clusters_n{n_groups}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    feature_cols = [
        "sunlight", "carbon_per_panel", "median_income", "black_prop",
        "panel_utilization", "coeff_mean", "coeff_std", "coeff_p90", "state_cutoff_final"
    ]
    X = zip_feature_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    Xs = StandardScaler().fit_transform(X)

    kmeans = MiniBatchKMeans(n_clusters=n_groups, random_state=random_state, n_init=10, batch_size=1024)
    labels = kmeans.fit_predict(Xs)

    out = zip_feature_df[["zip", "state_code"]].copy()
    out["group_id"] = labels.astype(int)
    out.to_csv(cache_path, index=False)
    return out

def compute_metrics_from_placements(placements):
    vals = {}
    for obj in objectives:
        vals[obj.name] = float(obj.calc(zips_df, placements))
    return vals

def evaluate_strategy(zip_feature_df, cluster_df, incentives_by_group):
    merged = zip_feature_df[["zip", "state_code", "count_qualified", "payload"]].merge(cluster_df, on=["zip", "state_code"], how="inner")
    placements = {}
    total_spending = 0.0
    total_panels = 0.0

    for row in merged.itertuples(index=False):
        incentive = incentives_by_group[int(row.group_id)]
        threshold = -incentive
        cutoff = state_cutoffs[row.state_code][-1]
        required = required_incentives_from_payload(row.payload, cutoff)
        adoption = float(np.mean(required <= threshold))
        panels = max(0.0, adoption * float(row.count_qualified))
        placements[int(row.zip)] = panels
        total_panels += panels
        total_spending += incentive * panels

    metric_vals = compute_metrics_from_placements(placements)
    return {
        "Carbon Offset": metric_vals.get("Carbon Offset", np.nan),
        "Energy Generation": metric_vals.get("Energy Potential", np.nan),
        "Racial Equity": metric_vals.get("Racial Equity", np.nan),
        "Income Equity": metric_vals.get("Income Equity", np.nan),
        "Panels": total_panels,
        "Spending": total_spending,
        "placements": placements,
    }

def is_dominated(a, b):
    # maximize 5 metrics, minimize spending
    maximize_keys = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels"]
    better_or_equal = all(b[k] >= a[k] for k in maximize_keys) and b["Spending"] <= a["Spending"]
    strictly_better = any(b[k] > a[k] for k in maximize_keys) or b["Spending"] < a["Spending"]
    return better_or_equal and strictly_better

def pareto_filter(records):
    keep = []
    for i, rec in enumerate(records):
        dom = False
        for j, other in enumerate(records):
            if i == j:
                continue
            if is_dominated(rec, other):
                dom = True
                break
        if not dom:
            keep.append(rec)
    return keep

In [ ]:
def normalize_for_weighted_sum(rows):
    cols = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels", "Spending"]
    mat = pd.DataFrame([{k: r[k] for k in cols} for r in rows])
    mins = mat.min()
    maxs = mat.max()

    def score(rec, w):
        s = 0.0
        for k in ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels"]:
            denom = max(maxs[k] - mins[k], 1e-9)
            s += w[k] * ((rec[k] - mins[k]) / denom)
        denom_s = max(maxs["Spending"] - mins["Spending"], 1e-9)
        s += w["Spending"] * ((maxs["Spending"] - rec["Spending"]) / denom_s)
        return s

    return score

def random_strategy(n_groups):
    return tuple(int(np.random.choice(INCENTIVE_GRID)) for _ in range(n_groups))

def neighbors(strategy):
    out = []
    for i in range(len(strategy)):
        idx = INCENTIVE_GRID.index(strategy[i])
        for shift in (-1, 1):
            j = idx + shift
            if 0 <= j < len(INCENTIVE_GRID):
                s = list(strategy)
                s[i] = INCENTIVE_GRID[j]
                out.append(tuple(s))
    return out

def run_weighted_stage(zip_feature_df, cluster_df, n_groups):
    eval_cache = {}
    pool = []

    # Seed sample for normalization.
    seed_strategies = [random_strategy(n_groups) for _ in range(150)]
    for s in tqdm(seed_strategies, desc=f"Seed eval n={n_groups}", leave=False):
        if s not in eval_cache:
            eval_cache[s] = evaluate_strategy(zip_feature_df, cluster_df, s)
        pool.append({"strategy": s, **eval_cache[s]})

    for _ in range(WEIGHT_SAMPLES):
        wv = np.random.dirichlet(np.ones(6))
        w = {
            "Carbon Offset": wv[0],
            "Energy Generation": wv[1],
            "Racial Equity": wv[2],
            "Income Equity": wv[3],
            "Panels": wv[4],
            "Spending": wv[5],
        }
        scorer = normalize_for_weighted_sum(pool)
        curr = random_strategy(n_groups)
        if curr not in eval_cache:
            eval_cache[curr] = evaluate_strategy(zip_feature_df, cluster_df, curr)

        best = curr
        best_score = scorer(eval_cache[curr], w)

        for _ in range(LOCAL_SEARCH_STEPS):
            neigh = neighbors(best)
            random.shuffle(neigh)
            improved = False
            for cand in neigh[:LOCAL_SEARCH_NEIGHBORS]:
                if cand not in eval_cache:
                    eval_cache[cand] = evaluate_strategy(zip_feature_df, cluster_df, cand)
                sc = scorer(eval_cache[cand], w)
                if sc > best_score:
                    best = cand
                    best_score = sc
                    improved = True
            if not improved:
                break

        pool.append({"strategy": best, **eval_cache[best]})

    # Deduplicate by strategy and filter Pareto.
    uniq = {}
    for r in pool:
        uniq[r["strategy"]] = r
    return pareto_filter(list(uniq.values())), eval_cache

def mutate_strategy(strategy):
    s = list(strategy)
    for i in range(len(s)):
        if random.random() < REFINE_MUTATION_RATE:
            idx = INCENTIVE_GRID.index(s[i])
            step = random.choice([-1, 1])
            idx2 = min(max(idx + step, 0), len(INCENTIVE_GRID) - 1)
            s[i] = INCENTIVE_GRID[idx2]
    return tuple(s)

def run_refinement_stage(zip_feature_df, cluster_df, n_groups, seed_front, eval_cache):
    pop = [r["strategy"] for r in seed_front[:REFINE_POP_SIZE]]
    while len(pop) < REFINE_POP_SIZE:
        pop.append(random_strategy(n_groups))

    for _ in tqdm(range(REFINE_GENERATIONS), desc=f"Refine n={n_groups}", leave=False):
        offspring = []
        for s in pop:
            child = mutate_strategy(s)
            offspring.append(child)
        cand = pop + offspring

        evaluated = []
        for s in cand:
            if s not in eval_cache:
                eval_cache[s] = evaluate_strategy(zip_feature_df, cluster_df, s)
            evaluated.append({"strategy": s, **eval_cache[s]})

        front = pareto_filter(evaluated)
        # Keep population by alternating front + random diversity.
        front_strats = [r["strategy"] for r in front]
        random.shuffle(front_strats)
        pop = front_strats[:REFINE_POP_SIZE]
        while len(pop) < REFINE_POP_SIZE:
            pop.append(random_strategy(n_groups))

    final_eval = []
    for s in set(pop):
        if s not in eval_cache:
            eval_cache[s] = evaluate_strategy(zip_feature_df, cluster_df, s)
        final_eval.append({"strategy": s, **eval_cache[s]})
    return pareto_filter(final_eval)

In [ ]:
all_front_rows = []
run_summary = []

for n_groups in tqdm(N_VALUES, desc="Run per-n optimization", unit="n"):
    t0 = time.time()
    cluster_df = cluster_zip_codes_for_n(zip_df, n_groups, EXPERIMENT_CACHE_DIR, random_state=RANDOM_SEED)

    seed_front, eval_cache = run_weighted_stage(zip_df, cluster_df, n_groups)
    refined_front = run_refinement_stage(zip_df, cluster_df, n_groups, seed_front, eval_cache)
    final_front = pareto_filter(seed_front + refined_front)

    rows = []
    for r in final_front:
        rows.append(
            {
                "n_groups": n_groups,
                "strategy": json.dumps(list(r["strategy"])),
                "Carbon Offset": r["Carbon Offset"],
                "Energy Generation": r["Energy Generation"],
                "Racial Equity": r["Racial Equity"],
                "Income Equity": r["Income Equity"],
                "Panels": r["Panels"],
                "Spending": r["Spending"],
            }
        )

    front_df = pd.DataFrame(rows).sort_values("Spending").reset_index(drop=True)
    front_path = RESULTS_DIR / f"pareto_front_n{n_groups}.csv"
    front_df.to_csv(front_path, index=False)
    all_front_rows.extend(rows)

    run_summary.append(
        {
            "n_groups": n_groups,
            "pareto_points": len(front_df),
            "elapsed_seconds": time.time() - t0,
            "output_file": str(front_path),
        }
    )

all_front_df = pd.DataFrame(all_front_rows)
summary_df = pd.DataFrame(run_summary)
all_front_df.to_csv(RESULTS_DIR / "pareto_front_all_n.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "pareto_run_summary.csv", index=False)

display(summary_df)
display(all_front_df.head())

In [ ]:
if all_front_df.empty:
    raise RuntimeError("No Pareto points found. Try increasing search budget.")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for n_groups, grp in all_front_df.groupby("n_groups"):
    axes[0, 0].scatter(grp["Spending"], grp["Carbon Offset"], label=f"n={n_groups}", alpha=0.75)
    axes[0, 1].scatter(grp["Spending"], grp["Energy Generation"], label=f"n={n_groups}", alpha=0.75)
    axes[1, 0].scatter(grp["Racial Equity"], grp["Income Equity"], label=f"n={n_groups}", alpha=0.75)
    axes[1, 1].scatter(grp["Panels"], grp["Carbon Offset"], label=f"n={n_groups}", alpha=0.75)

axes[0, 0].set_title("Carbon Offset vs Spending")
axes[0, 1].set_title("Energy Generation vs Spending")
axes[1, 0].set_title("Racial vs Income Equity")
axes[1, 1].set_title("Carbon Offset vs Panels")

axes[0, 0].set_xlabel("Spending")
axes[0, 0].set_ylabel("Carbon Offset")
axes[0, 1].set_xlabel("Spending")
axes[0, 1].set_ylabel("Energy Generation")
axes[1, 0].set_xlabel("Racial Equity")
axes[1, 0].set_ylabel("Income Equity")
axes[1, 1].set_xlabel("Panels")
axes[1, 1].set_ylabel("Carbon Offset")

for ax in axes.flatten():
    ax.grid(True, alpha=0.3)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=min(len(N_VALUES), 6))
plt.tight_layout()
plt.show()

display(all_front_df.groupby("n_groups")[["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity", "Panels"]].max())